In [ ]:
!pip install --upgrade transformers tokenizers

In [ ]:
# --- Imports ---
import torch
import torch.nn as nn
import math
import pandas as pd
import re
import os
from sklearn.model_selection import train_test_split
# We still need the tokenizer from transformers
from transformers import BertTokenizer

In [ ]:
# --- Component 1: Multi-Head Attention ---
class MultiHeadAttention(nn.Module):
    def __init__(self, dim, n_heads):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads = n_heads
        self.d_k = dim // n_heads # Head dimension
        self.dim = dim

        # Simple linear layers for projections
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim) # Final output projection

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)

        # 1. Project and split into heads
        # (batch, seq, dim) -> (batch, n_heads, seq, d_k)
        q = self.q_proj(q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        k = self.k_proj(k).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        v = self.v_proj(v).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        # 2. Scaled dot-product attention
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)

        if mask is not None:
            # Apply the mask (e.g., for padding)
            scores = scores.masked_fill(mask == 0, -1e9)

        attn_weights = torch.softmax(scores, dim=-1)

        # 3. Get context vector
        context = attn_weights @ v

        # 4. Combine heads and do final projection
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.dim)
        return self.out_proj(context)

# --- Component 2: Feed-Forward Network ---
class FeedForward(nn.Module):
    def __init__(self, dim, ff_dim, dropout=0.1):
        super().__init__()
        # Use nn.Sequential for a common FFN pattern
        self.net = nn.Sequential(
            nn.Linear(dim, ff_dim),
            nn.GELU(), # BERT uses GELU
            nn.Dropout(dropout),
            nn.Linear(ff_dim, dim)
        )

    def forward(self, x):
        return self.net(x)

# --- Component 3: Transformer Encoder Layer ---
class EncoderLayer(nn.Module):
    def __init__(self, dim, n_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(dim, n_heads)
        self.ff = FeedForward(dim, ff_dim, dropout)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

        # Flag for assignment reporting
        self.report_shapes = False

    def forward(self, x, mask=None):
        # Attention block (self-attention, so Q, K, V are all 'x')
        attn_out = self.attn(x, x, x, mask)

        # This is the "report" part, prints during forward pass
        if self.report_shapes:
            print(f"[Report] Shape of Output (Multi-Head Attention): {attn_out.shape}")

        x = self.norm1(x + self.dropout(attn_out)) # Add & Norm

        # Feed-forward block
        ff_out = self.ff(x)

        if self.report_shapes:
            print(f"[Report] Shape of Output (Feed-Forward): {ff_out.shape}")

        x = self.norm2(x + self.dropout(ff_out)) # Add & Norm
        return x

# --- BERT Embeddings ---
class BertEmbeddings(nn.Module):
    def __init__(self, vocab_size, dim, max_len, n_segments):
        super().__init__()
        self.tok_embed = nn.Embedding(vocab_size, dim, padding_idx=0)
        self.pos_embed = nn.Embedding(max_len, dim)
        self.seg_embed = nn.Embedding(n_segments, dim)

        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(0.1)

        # A common way to store non-parameter tensors
        self.register_buffer("position_ids", torch.arange(max_len).expand((1, -1)))

    def forward(self, input_ids, segment_ids):
        seq_len = input_ids.size(1)

        tok_emb = self.tok_embed(input_ids)
        # Use the pre-computed position_ids
        pos_emb = self.pos_embed(self.position_ids[:, :seq_len])
        seg_emb = self.seg_embed(segment_ids)

        # Sum all three embeddings
        embeddings = tok_emb + pos_emb + seg_emb

        # Final norm and dropout
        return self.dropout(self.norm(embeddings))

# --- Main BERT Model Class ---
class MyBertModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config # Store config

        self.embeddings = BertEmbeddings(
            config['vocab_size'],
            config['dim'],
            config['max_len'],
            config['n_segments']
        )

        # Stack of encoder layers
        self.layers = nn.ModuleList(
            [EncoderLayer(config['dim'], config['n_heads'], config['ff_dim'])
             for _ in range(config['n_layers'])]
        )

        # Pooler takes the [CLS] token output
        self.pooler = nn.Sequential(
            nn.Linear(config['dim'], config['dim']),
            nn.Tanh()
        )

        # Final classifier head
        self.classifier = nn.Linear(config['dim'], config['num_classes'])

    def forward(self, input_ids, segment_ids, attention_mask=None):

        # Create the 4D attention mask for the multi-head attention
        if attention_mask is not None:
            # (batch, seq) -> (batch, 1, 1, seq)
            mask = attention_mask.unsqueeze(1).unsqueeze(2)
        else:
            mask = None

        # 1. Get Embeddings
        x = self.embeddings(input_ids, segment_ids)
        if self.config.get('report_shapes', False): # Check config flag
            print(f"[Report] Shape of Token Embeddings: {x.shape}")

        # 2. Pass through Encoder layers
        for layer in self.layers:
            x = layer(x, mask=mask)

        if self.config.get('report_shapes', False):
            print(f"[Report] Shape of Final Encoder Output (Layer {self.config['n_layers']}): {x.shape}")

        # 3. Pooler
        # Get the hidden state of the first token ([CLS])
        cls_token_state = x[:, 0]
        pooled_output = self.pooler(cls_token_state)

        # 4. Classifier
        logits = self.classifier(pooled_output)

        return logits

    # Helper function to turn on printing for the report
    def set_report_shapes(self, report=True):
        self.config['report_shapes'] = report
        for layer in self.layers:
            layer.report_shapes = report


# -----------------------------------------------------------------
# Task 1, Step 3 & 4: Testing and Report
# -----------------------------------------------------------------

print("--- [Task 1] Starting Test and Report ---")

# --- 1. Define Model Config ---
# Assignment specs
bert_config = {
    'dim': 768,
    'n_heads': 12,
    'n_layers': 2,
    'ff_dim': 768 * 4, # 3072
    'num_classes': 2,   # For toxicity

    # Standard bert-base-uncased specs
    'vocab_size': 30522,
    'max_len': 512,
    'n_segments': 2,
}

# --- 2. Initialize Tokenizer ---
print("Loading 'bert-base-uncased' tokenizer...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# --- 3. Define and Tokenize Sample Sentence ---
sentence = "This is a sample sentence with more than 10 words, which we will use to test our BERT model."
print(f"Sample Sentence: '{sentence}'")

# Tokenize it (pad to 64 for a clean example)
inputs = tokenizer(
    sentence,
    return_tensors='pt',
    max_length=64,
    padding='max_length',
    truncation=True
)

input_ids = inputs['input_ids']
attn_mask = inputs['attention_mask']
# Per assignment, pass 0 for all segment IDs
seg_ids = torch.zeros_like(input_ids)

print(f"Tokenized 'input_ids' shape: {input_ids.shape}")

# --- 4. Instantiate and Test Model ---
print("\nInstantiating custom MyBertModel...")
model = MyBertModel(bert_config)
model.eval() # Set to evaluation mode

# ---------------------------------------------------
# THIS IS THE KEY STEP FOR THE REPORT
# Turn on the internal print() statements
model.set_report_shapes(True)
# ---------------------------------------------------

print("\n--- [Report] Shapes of Embeddings and Outputs (from forward pass) ---")
with torch.no_grad(): # Don't calculate gradients
    # Run the forward pass. This will trigger the print() statements.
    output_probs = model(input_ids, seg_ids, attn_mask)

print(f"[Report] Shape of Output Probabilities (Classifier): {output_probs.shape}")

# --- 5. Report: Shape of each parameter ---
print("\n--- [Report] Shape of Each Parameter ---")
total_params = 0
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name}: \t{param.shape}")
        total_params += param.numel()

print(f"\nTotal Model Parameters: {total_params / 1_000_000:.2f}M")
print("(Note: A full BERT-base has ~110M params; ours is smaller with 2 layers)")
print("\n--- [Task 1] Test and Report Complete ---")

--- [Task 1] Starting Test and Report ---
Loading 'bert-base-uncased' tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Sample Sentence: 'This is a sample sentence with more than 10 words, which we will use to test our BERT model.'
Tokenized 'input_ids' shape: torch.Size([1, 64])

Instantiating custom MyBertModel...

--- [Report] Shapes of Embeddings and Outputs (from forward pass) ---
[Report] Shape of Token Embeddings: torch.Size([1, 64, 768])
[Report] Shape of Output (Multi-Head Attention): torch.Size([1, 64, 768])
[Report] Shape of Output (Feed-Forward): torch.Size([1, 64, 768])
[Report] Shape of Output (Multi-Head Attention): torch.Size([1, 64, 768])
[Report] Shape of Output (Feed-Forward): torch.Size([1, 64, 768])
[Report] Shape of Final Encoder Output (Layer 2): torch.Size([1, 64, 768])
[Report] Shape of Output Probabilities (Classifier): torch.Size([1, 2])

--- [Report] Shape of Each Parameter ---
embeddings.tok_embed.weight: 	torch.Size([30522, 768])
embeddings.pos_embed.weight: 	torch.Size([512, 768])
embeddings.seg_embed.weight: 	torch.Size([2, 768])
embeddings.norm.weight: 	torch.Size([768])

In [ ]:

FILE_PATH = r"D:\NLP\assignment 3\all_comments.tsv"
# ----------------------------------------------------

# --- Script settings ---
MIN_SAMPLES = 2500  # "at least 2500 each"
TEST_SIZE = 0.2
SEED = 42 # for reproducible splits/sampling

# --- Columns we need from Fakeddit ---
# These are the standard names
TEXT_COLS = ['title', 'clean_comment']
LABEL_COL = '2_way_label'

print("--- [Task 2] Starting Fakeddit Preprocessing ---")

# --- Load Data ---
try:
    # Fakeddit uses TABS, not commas. This is important.
    df = pd.read_csv(FILE_PATH, sep='\t')
    print(f"Loaded {len(df)} rows from file.")

    # Check for needed columns
    required_cols = TEXT_COLS + [LABEL_COL]
    if not all(col in df.columns for col in required_cols):
        print(f"Error: Your file is missing one of these columns: {required_cols}")
        print(f"Found columns: {df.columns.tolist()}")
        raise ValueError("Missing required Fakeddit columns.")

except FileNotFoundError:
    print(f"ERROR: File not found at '{FILE_PATH}'")
    print("Please fix the path variable at the top of the script and re-run.")
    raise
except Exception as e:
    print(f"Error loading data: {e}")
    raise

original_count = len(df)

# --- Step 1: Create a single 'text' input ---
# We'll combine title and comment.
# .fillna('') handles posts that have a title but no comment text.
print("Step 1: Combining 'title' and 'clean_comment' into 'text' column.")
df['text'] = df['title'] + " " + df['clean_comment'].fillna('')

# --- Step 2 (BONUS): Data Cleaning (Remove Duplicates) ---
count_before = len(df)
# Drop rows where the combined text is identical
df.drop_duplicates(subset=['text'], inplace=True)
count_after = len(df)
print(f"Step 2 (Bonus): Removed {count_before - count_after} duplicate posts.")

# --- Step 3: Balance the Dataset ---
print(f"Step 3: Balancing to {MIN_SAMPLES} real and {MIN_SAMPLES} fake...")

# Get the counts *after* cleaning
# 2_way_label: 0 = Real, 1 = Fake
real_count = (df[LABEL_COL] == 0).sum()
fake_count = (df[LABEL_COL] == 1).sum()
print(f"        (Cleaned counts: {real_count} real, {fake_count} fake)")

# Check if we have enough data
if fake_count < MIN_SAMPLES or real_count < MIN_SAMPLES:
    print(f"Warning: Not enough samples. Using {min(fake_count, real_count)} instead.")
    MIN_SAMPLES = min(fake_count, real_count)

# Separate the two classes
real_df = df[df[LABEL_COL] == 0]
fake_df = df[df[LABEL_COL] == 1]

# Sample from each
real_sample = real_df.sample(n=MIN_SAMPLES, random_state=SEED)
fake_sample = fake_df.sample(n=MIN_SAMPLES, random_state=SEED)

# Combine and shuffle
df_balanced = pd.concat([real_sample, fake_sample])
df_balanced = df_balanced.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"        Created balanced set of {len(df_balanced)} rows.")

# --- Step 4: Train-Test Split (80/20) ---
print(f"Step 4: Splitting into 80% train / 20% test...")

# We will use 'text' as our input (X) and '2_way_label' as our output (y)
train_df, test_df = train_test_split(
    df_balanced,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df_balanced[LABEL_COL] # Keep the 50/50 balance
)

print(f"        Train: {len(train_df)} rows | Test: {len(test_df)} rows")

# --- Step 5: Report Statistics (Adapted from assignment) ---
print("\n--- [Task 2] Final Statistics Report ---")
print(f"1. Number of posts in original dataset: {original_count}")
print(f"2. Number of real/fake posts (after cleaning):")
print(f"   - Real ({LABEL_COL}=0): {real_count}")
print(f"   - Fake ({LABEL_COL}=1): {fake_count}")
print(f"3. Number of real/fake in the *balanced* dataset built:")
print(f"   - Real ({LABEL_COL}=0): {len(real_sample)}")
print(f"   - Fake ({LABEL_COL}=1): {len(fake_sample)}")
print(f"   - Total:     {len(df_balanced)}")
print(f"4. Number of posts in final train and test sets:")
print(f"   - Train Set: {len(train_df)}")
print(f"   - Test Set:  {len(test_df)}")
print("--- [Task 2] Preprocessing Complete ---")

--- [Task 2] Starting Fakeddit Preprocessing ---
ERROR: File not found at 'D:\NLP\assignment 3\all_comments.tsv'
Please fix the path variable at the top of the script and re-run.


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\NLP\\assignment 3\\all_comments.tsv'